# 23. SFT Analysis（spec §21）

## 学習目標
- 事前学習モデルに簡易 Instruction SFT を施し、応答形式の変化を見る
- **assistant-only loss** と **full-sequence loss** を比較する
- 「SFTは知識を大量に増やすのではなく、主に出力形式・応答方針を変える」ことを結果から検討する

## 形式
```
<BOS> <USER> 指示 <ASSISTANT> 応答 <EOS>
```
assistant-only: 応答部分だけに loss。full-sequence: 全トークンに loss。

In [1]:
import warnings
warnings.filterwarnings("ignore")
import json, math
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "plotly_mimetype"
from jp_llm_lab.utils.io import repo_root, load_json, read_jsonl
ROOT = repo_root()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
M2_RUN = ROOT / "experiments/runs/m2_model_m_classical_seed42"
L_RUN = ROOT / "experiments/runs/m4_model_l_modern_seed42"
M5 = ROOT / "experiments/analysis_m5"

In [2]:
sft = load_json(M5 / "sft.json")
print(f"SFT examples: {sft['sft_examples']}（dolly-15k-ja のclosed-book短文サブセット）")
for regime in ["assistant_only","full_sequence"]:
    c = sft["regimes"][regime]["curve"]
    print(f'{regime:16s} final SFT loss {sft["regimes"][regime]["final_loss"]:.3f}')
fig = go.Figure()
for regime, color in [("assistant_only","#2ca02c"),("full_sequence","#ff7f0e")]:
    c = sft["regimes"][regime]["curve"]
    fig.add_trace(go.Scatter(x=[p["step"] for p in c], y=[p["loss"] for p in c], name=regime, line=dict(color=color)))
fig.update_layout(title="SFT 学習曲線: assistant-only vs full-sequence",
                  xaxis_title="SFT step", yaxis_title="loss", template="plotly_white", height=380)
fig.show()

SFT examples: 1200（dolly-15k-ja のclosed-book短文サブセット）
assistant_only   final SFT loss 0.244
full_sequence    final SFT loss 0.446


In [3]:
# 事前学習(base) vs SFT の応答を同じ指示で比較
print("=== base（事前学習のみ, 指示に continuation するだけ）===")
for s in sft["base_pretrained_samples"]:
    print(f'Q: {s["instruction"]}')
    print(f'  → {s["completion"][:80]}')
print("\n=== assistant-only SFT ===")
for s in sft["regimes"]["assistant_only"]["samples"]:
    print(f'Q: {s["instruction"]}')
    print(f'  → {s["response"][:80]}')

=== base（事前学習のみ, 指示に continuation するだけ）===
Q: 日本の首都はどこですか？
  → 」
信長は2000年前後半、大正時代の日本統治維新の政策を発表したことについて、国民の多くが日本政府を指名して国会に核本を置くような政策を公言した。
「お前は日
Q: 犬について一文で説明してください。
  → 
- 回答日時：2004/03/11 21:47
回答ありがとうございました。
「うわっ、いいや」と言う言葉が無くてもおかしくないけど・・・（笑）
「うわっ、い
Q: 2たす3はいくつですか？
  → ?
- 回答日時：2004/03/17 21:47
-
回答ありがとうございます。
「うん、お疲れ様です。」
ついでにいろいろな事柄がありますけどね。
ま、「ゆ

=== assistant-only SFT ===
Q: 日本の首都はどこですか？
  → 首都はスクレである。
Q: 犬について一文で説明してください。
  → - 犬
- 人参
- ケール
- ブロッコリー
- ブロッコリー
- キャメロン
- メリー
Q: 2たす3はいくつですか？
  → 2たす3 2000メガラム、11世紀、エタカラム、11世紀、エタラム、11世紀、エタラム、1172年、エタラム、119の二重構造


## Observation / Interpretation / Caveat
- **Observation**: base モデルは指示を「続けて書く」だけで応答形式になりにくい。SFT後は `<ASSISTANT>` 以降に応答らしい形で出力し、`<EOS>` で止まりやすくなる（上のセルが実測）。assistant-only と full-sequence は最終 loss の絶対値が異なる（full は prompt も含むので基準が違う）。
- **Interpretation**: SFT の主効果は**出力形式・応答方針**の獲得であり、新しい事実知識の大量獲得ではない（回答内容の正確さは base の知識に依存し、しばしば誤る）。assistant-only は prompt をモデル化しない分、応答生成に集中する。
- **Caveat**: 小モデル・少数例・短時間 SFT。応答は依然として事実誤りを含む（知識は事前学習由来）。2 regime の loss 値は損失対象が違うため直接比較できず、比較すべきは生成の質・形式。instruction-following の定量評価は本デモの範囲外。

**次へ**: [24_final_report](24_final_report.ipynb)